In [1]:
"""
The evaluation API requires that you set up a server which will respond to inference requests.
We have already defined the server; you just need write the predict function.
When we evaluate your submission on the hidden test set the client defined in `nfl_gateway` will run in a different container
with direct access to the hidden test set and hand off the data timestep by timestep.
Your code will always have access to the published copies of the copmetition files.
"""

import os
import numpy as np
import pandas as pd
import polars as pl
import tensorflow as tf


import sys

import os
import gdown 
import zipfile 
from scipy import stats
# Set the default style for seaborn
import seaborn as sns
sns.set(style="whitegrid")      


import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import display, Image
import sklearn 



import time
import warnings 

    
# Librerías ML CPU
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optuna para bayesian optimization
import optuna

# tensor flow 
import tensorflow as Tf


from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils.validation import check_is_fitted


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import kaggle_evaluation.nfl_inference_server


# models 
import joblib
from datetime import datetime





# DATA 
data_path = '/kaggle/input/nfl-big-data-bowl-2026-prediction/train'# Ruta base donde se almacenan los archivos CSV

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
input_dfs = []
for week in range(1, 19):
    fname = os.path.join(data_path, f'input_2023_w{week:02d}.csv')
    if os.path.exists(fname):
        input_dfs.append(pd.read_csv(fname))
        print(f'Archivo {fname} cargado. Observaciones: {input_dfs[-1].shape[0]}')
    else:
        print(f'  No se encontró {fname}')

input_df = pd.concat(input_dfs, ignore_index=True) # The input df is a pandas dataframe with all weeks data 



output_dfs = []
for week in range(1, 19):
    fname = os.path.join(data_path, f'output_2023_w{week:02d}.csv')
    if os.path.exists(fname):
        output_dfs.append(pd.read_csv(fname))
        print(f'Archivo {fname} cargado. Observaciones: {input_dfs[-1].shape[0]}')

    else:
        print(f'  No se encontró {fname}')

if output_dfs:
    output_df = pd.concat(output_dfs, ignore_index=True)
else:
    output_df = pd.DataFrame()



########

# Preprocesamiento

def parse_height(h):
    if isinstance(h, str) and '-' in h:
        ft, inch = h.split('-')
        return int(ft) * 12 + int(inch)
    return np.nan



input_df['player_height'] = input_df['player_height'].apply(parse_height)
input_df['player_birth_date'] = pd.to_datetime(input_df['player_birth_date'], errors='coerce')
reference_date = pd.to_datetime('2025-11-19')
input_df['age'] = (reference_date - input_df['player_birth_date']).dt.days / 365.25


numerical_features = ['x','y','s','a','o','dir','player_weight','absolute_yardline_number','ball_land_x', 'ball_land_y' , 'player_height', 'age' ]
categorical_features = ['player_position', 'player_side', 'player_role', 'play_direction']



numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

############

# model data

target_cols = ['x', 'y']

 
merge_cols = ['game_id', 'play_id', 'nfl_id', 'frame_id']


data_full = input_df.merge(
    output_df[merge_cols + target_cols], # enrutar con el frame especifico a predecir 
    on=merge_cols, # unir por las columnas clave 
    how='inner',
    suffixes=('', '_out')  # deja las features sin sufijo y marca los targets
)
dataToPredict = ['x_out', 'y_out']

X = data_full[numerical_features + categorical_features]
y = data_full[dataToPredict] 

X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size= 0.6,  test_size=0.4, random_state=42)


X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp , test_size=0.5, random_state=42)

print('Tamaños de los subconjuntos:')
print('Train :', X_train.shape)
print('Val   :', X_val.shape)
print('Test  :', X_test.shape)

#######

# Preprocesado

# Ajustamos el preprocesador y generamos matrices listas para Keras
preprocessor_dl = preprocessor  # reutiliza la misma configuración definida arriba
preprocessor_dl.fit(X_train)

X_train_proc = preprocessor_dl.transform(X_train)
X_val_proc = preprocessor_dl.transform(X_val)
X_test_proc = preprocessor_dl.transform(X_test)

# Convertir a matrices densas y tipo float32
if hasattr(X_train_proc, "toarray"):
    X_train_proc = X_train_proc.toarray()
    X_val_proc = X_val_proc.toarray()
    X_test_proc = X_test_proc.toarray()

X_train_proc = X_train_proc.astype('float32')
X_val_proc = X_val_proc.astype('float32')
X_test_proc = X_test_proc.astype('float32')

y_train_proc = y_train[['x_out', 'y_out']].to_numpy(dtype='float32')
y_val_proc = y_val[['x_out', 'y_out']].to_numpy(dtype='float32')
y_test_proc = y_test[['x_out', 'y_out']].to_numpy(dtype='float32')

# Submuestreo opcional para reducir costo (ajusta quick_frac=1.0 para entrenamiento completo)
quick_frac = 0.02
if 0 < quick_frac < 1:
    n = int(len(X_train_proc) * quick_frac)
    idx = np.random.permutation(len(X_train_proc))[:max(1000, n)]
    X_train_dl = X_train_proc[idx]
    y_train_dl = y_train_proc[idx]
else:
    X_train_dl = X_train_proc
    y_train_dl = y_train_proc

X_val_dl = X_val_proc
X_test_dl = X_test_proc
y_val_dl = y_val_proc
y_test_dl = y_test_proc

''' np.savez('data/prep_data_dl.npz',
         X_train=X_train_proc,
         y_train=y_train_proc,
         X_val=X_val_proc,
         y_val=y_val_proc,
         X_test=X_test_proc,
         y_test=y_test_proc)
'''
print(f'Datos DL -> train {X_train_dl.shape}, val {X_val_dl.shape}, test {X_test_dl.shape}')
print(f'NaNs en X_train_dl: {np.isnan(X_train_dl).sum()} | NaNs en y_train_dl: {np.isnan(y_train_dl).sum()}')

# data = np.load('data/prep_data_dl.npz')
X_train_dl = X_train_proc.astype('float32')
y_train_dl = y_train_proc.astype('float32')
X_val_dl   = X_val_proc.astype('float32')
y_val_dl   = y_val_proc.astype('float32')
X_test_dl  = X_test_proc.astype('float32')
y_test_dl  = y_test_proc.astype('float32')

# dimensión del vector de características
feature_dim = X_train_dl.shape[1]

# número de épocas y tamaño de lote para todos los modelos
epochs_dl = 30
batch_size_dl = 256

# callbacks comunes: parada temprana y terminación en NaN
common_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.TerminateOnNaN()
]

print("ready for modeling with best model found")

######
# MODEL BEST MODEL

mlp = tf.keras.Sequential(name='mlp_denso')
mlp.add(tf.keras.layers.Input(shape=(feature_dim,)))
mlp.add(tf.keras.layers.Dense(128, activation='relu'))
mlp.add(tf.keras.layers.Dense(64, activation='relu'))
mlp.add(tf.keras.layers.Dense(32, activation='relu'))
mlp.add(tf.keras.layers.Dense(2, activation='linear', dtype='float32'))
mlp.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='mse', metrics=['mae'])

mlp.fit(X_train_dl, y_train_dl,
        validation_data=(X_val_dl, y_val_dl),
        epochs=epochs_dl,
        batch_size=batch_size_dl,
        callbacks=common_callbacks,
        verbose=1)

#####################
# Predicciones y métricas
mlp_val_pred  = mlp.predict(X_val_dl,  batch_size=batch_size_dl)
mlp_test_pred = mlp.predict(X_test_dl, batch_size=batch_size_dl)



def metricas_basicas(y_true, y_pred):
    # MAE
    mae = float(np.mean(np.abs(y_true - y_pred)))

    # MSE
    mse = float(np.mean((y_true - y_pred)**2))

    # RMSE
    rmse = float(np.sqrt(mse))

    # R²
    ss_res = float(np.sum((y_true - y_pred)**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true, axis=0))**2))
    r2 = float(1 - ss_res / (ss_tot + 1e-9))

    # MAPE
    mape = float(np.mean(
        np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))
    ) * 100)

    return mae, rmse, r2, mape

# Calcula métricas por modelo y conjunto
dl_runs = []
for modelo, preds_val, preds_test in [
    ('MLP denso',        mlp_val_pred,  mlp_test_pred)
]:
    for tag, y_true, y_pred in [('val', y_val_dl, preds_val), ('test', y_test_dl, preds_test)]:
        mae, rmse, r2, mape = metricas_basicas(y_true, y_pred)
        dl_runs.append([modelo, tag, mae, rmse, r2, mape])

dl_results_df = pd.DataFrame(dl_runs, columns=['Modelo', 'Conjunto', 'MAE', 'RMSE', 'R2', 'MAPE (%)'])

# Selecciona solo métricas de validación de los DL
dl_val = dl_results_df[dl_results_df['Conjunto'] == 'val'].copy()

# Mapea a la estructura que ya usabas en "results"
dl_results_list = [
[
    [row['Conjunto'], row['MAE'], row['RMSE'], row['R2'], row['MAPE (%)']]
    for _, row in dl_results_df[dl_results_df['Conjunto'] == 'val'].iterrows()
]
]

# Elige conjunto (val o test)
dl_scope = dl_results_df[dl_results_df['Conjunto'] == 'val']  # o 'test'

# Construye la tabla final (con nombre de modelo)
results = dl_scope[['Modelo', 'MAE', 'RMSE', 'R2', 'MAPE (%)']].values.tolist()


metrics_df = pd.DataFrame(results, columns=['Modelo', 'MAE', 'RMSE', 'R²', 'MAPE (%)'])
metrics_df = metrics_df.sort_values(by='R²', ascending=False).reset_index(drop=True)

save_path = 'model_performance_comparison.csv'
metrics_df.to_csv(save_path, index=False)
display(metrics_df)


model_dict = {
    'MLP denso': mlp

}

best_model_name = metrics_df.iloc[0]['Modelo']  # asumiendo metrics_df solo tiene DL
best_model = model_dict[best_model_name]
print(best_model_name, best_model)




X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)




# Reutiliza el preprocesador y densifica
X_train_full_proc = preprocessor_dl.transform(X_train_full)
if hasattr(X_train_full_proc, "toarray"):
    X_train_full_proc = X_train_full_proc.toarray()
X_train_full_proc = X_train_full_proc.astype("float32")

y_train_full_proc = y_train_full[['x_out', 'y_out']].to_numpy(dtype='float32')


# Transforma X_test con el mismo preprocesador que usaste para entrenar
X_test_proc = preprocessor_dl.transform(X_test)
if hasattr(X_test_proc, "toarray"):
    X_test_proc = X_test_proc.toarray()
X_test_proc = X_test_proc.astype("float32")

# Si necesitas y_test en numpy para métricas:
y_test_proc = y_test[['x_out', 'y_out']].to_numpy(dtype='float32')


print(X_train_full_proc.dtype, y_train_full_proc.dtype)  # debe ser float32



best_model.fit(X_train_full_proc, y_train_full_proc, batch_size=256, epochs=8)



y_test_pred_best_model = best_model.predict(X_test_proc)



mae_best_model = mean_absolute_error(y_test, y_test_pred_best_model)
mse_best_model = mean_squared_error(y_test, y_test_pred_best_model)

rmse_best_model= np.sqrt(mean_squared_error(y_test, y_test_pred_best_model))
r2_best_model = r2_score(y_test, y_test_pred_best_model)
mape_best_model = np.mean(np.abs((y_test - y_test_pred_best_model) / y_test)) * 100


print('\nPrueba best model:')
print(f'MAE : {mae_best_model:.4f}')
print(f'MSE : {mse_best_model:.4f}')
print(f'RMSE: {rmse_best_model:.4f}')
print(f'R2  : {r2_best_model:.4f}')
print(f'MAPE: {mape_best_model:.4f}%')

### formato entrega


test_input = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2026-prediction/test_input.csv') ### 

test_merged = test_input

test_merged['player_height'] = test_merged['player_height'].apply(parse_height)
test_merged['player_birth_date'] = pd.to_datetime(test_merged['player_birth_date'], errors='coerce')
reference_date = pd.to_datetime('2025-11-19')
test_merged['age'] = (reference_date - test_merged['player_birth_date']).dt.days / 365.25




available_features = [c for c in numerical_features + categorical_features if c in test_merged.columns]
missing_features = [c for c in numerical_features + categorical_features if c not in test_merged.columns]

if missing_features:
    print(f" Faltan columnas: {missing_features}")
    for col in missing_features:
        test_merged[col] = 0  # crear columnas vacías

test_features = test_merged[numerical_features + categorical_features]



expected_input_cols = list(preprocessor.feature_names_in_)

for col in expected_input_cols:
    if col not in test_merged.columns:
        test_merged[col] = 0

test_features = test_merged[expected_input_cols]

print(f" Alineadas: {len(test_features.columns)} columnas (esperadas: {len(expected_input_cols)})")



test_features_proc = preprocessor_dl.transform(test_features)
if hasattr(test_features_proc, "toarray"):
    test_features_proc = test_features_proc.toarray()
test_features_proc = test_features_proc.astype("float32")



submissionPreds = best_model.predict(test_features_proc)

predictions = pd.DataFrame({
    'x': submissionPreds[:, 0],
    'y': submissionPreds[:, 1]
})


def predict(test: pl.DataFrame, test_input: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    """Replace this function with your inference code.
    You can return either a Pandas or Polars dataframe, though Polars is recommended for performance.
    Each batch of predictions (except the very first) must be returned within 5 minutes of the batch features being provided.
    """
    predictions = pl.DataFrame({'x': [0.0] * len(test), 'y': [0.0] * len(test)})

    assert isinstance(predictions, (pd.DataFrame, pl.DataFrame))
    assert len(predictions) == len(test)
    return predictions


# When your notebook is run on the hidden test set, inference_server.serve must be called within 10 minutes of the notebook starting
# or the gateway will throw an error. If you need more than 15 minutes to load your model you can do so during the very
# first `predict` call, which does not have the usual 5 minute response deadline.
inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/nfl-big-data-bowl-2026-prediction/',))


2025-12-03 21:56:17.410654: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764798977.597300      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764798977.648042      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


/kaggle/input/nfl-big-data-bowl-2026-prediction/test_input.csv
/kaggle/input/nfl-big-data-bowl-2026-prediction/test.csv
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/nfl_inference_server.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/nfl_gateway.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/__init__.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/nfl-big-data-bowl-2026-prediction/kaggle_evaluation/core/generated/kaggle_evaluati

I0000 00:00:1764799012.799752      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30


I0000 00:00:1764799015.713060      63 service.cc:148] XLA service 0x7a398000b800 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764799015.713485      63 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764799015.945935      63 cuda_dnn.cc:529] Loaded cuDNN version 90300


  76/1314 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2264.0908 - mae: 40.1792

I0000 00:00:1764799016.614108      63 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1314/1314 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 413.0061 - mae: 10.7637 - val_loss: 16.6077 - val_mae: 2.9883
Epoch 2/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 16.3511 - mae: 2.9514 - val_loss: 15.8161 - val_mae: 2.9008
Epoch 3/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 15.5707 - mae: 2.8703 - val_loss: 15.2521 - val_mae: 2.8327
Epoch 4/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 15.1908 - mae: 2.8323 - val_loss: 15.0695 - val_mae: 2.8104
Epoch 5/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 14.9407 - mae: 2.8063 - val_loss: 14.9105 - val_mae: 2.7997
Epoch 6/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 14.8769 - mae: 2.7948 - val_loss: 15.0131 - val_mae: 2.8204
Epoch 7/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 14.5614 - mae: 2.7623 - val_loss: 14.4918 - val_mae: 2.7410
Epoch 8/30
1314/1314 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 14.4056 - mae: 2.7410 - val_loss: 14.3113 - val_mae: 2.7259
Epoch 9/30
1314/1314 ━━━━━━━━━━━━

,Modelo,MAE,RMSE,R²,MAPE (%)
0,MLP denso,2.687912,3.746483,0.965764,11.414931


MLP denso <Sequential name=mlp_denso, built=True>
float32 float32
Epoch 1/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 14.1133 - mae: 2.7062
Epoch 2/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 14.1219 - mae: 2.7046
Epoch 3/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 13.9607 - mae: 2.6907
Epoch 4/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 13.8994 - mae: 2.6870
Epoch 5/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 13.9145 - mae: 2.6871
Epoch 6/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 13.8246 - mae: 2.6803
Epoch 7/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 13.8677 - mae: 2.6817
Epoch 8/8
1752/1752 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 13.7663 - mae: 2.6719
3503/3503 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step

Prueba best model:
MAE : 2.6740
MSE : 13.7581
RMSE: 3.7092
R2  : 0.9492
MAPE: 11.2157%
 Alineadas: 16 columnas (esperadas: 16)
1555/1555 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
